In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import quote

import numpy as np

os.environ["POLARS_OOC_MEMORY_BUDGET_MB"] = "100"  # disable Polars' own memory budget
os.environ["POLARS_ENGINE_AFFINITY"] = "streaming"
os.environ["POLARS_STREAMING_CHUNK_SIZE"] = "2000"  # small chunks to stress memory
os.environ["POLARS_VERBOSE"] = "0"  # enable verbose logging to stdout for debugging
os.environ["POLARS_MAX_THREADS"] = "1"  # enable verbose logging to stdout for debugging

import polars as pl

_IMPL = Path(".").parent / "_memory_analysis_impl.py"
assert _IMPL.exists(), f"Expected {_IMPL} to exist"


def write_partitioned_measurements(src: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Write partitioned parquet with (measurement, unit, timestamp, value) columns.

    Each partition holds ``ROWS_PER_PARTITION`` readings spread over
    ``N_MEASUREMENTS`` distinct measurements, so grouping by measurement yields
    a handful of groups with very long ``timestamp``/``value`` list columns.
    """
    total_mbytes = 0
    for p in range(n_partitions):
        measurement = f"measurement_{p}"
        measurement_col = "measurement"
        partition_dir = src / f"{quote(measurement_col)}={quote(measurement)}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        df = pl.DataFrame([
            pl.repeat(measurement, rows_per_partition, eager=True).rename(measurement_col),
            (pl.int_range(0, rows_per_partition, eager=True) % 5)
            .cast(pl.String)
            .str.pad_start(length=3, fill_char="0")
            .rename("channel"),
            pl.Series(np.random.rand(rows_per_partition)).rename("value"),
        ])
        df.write_parquet(partition_dir / "00001.parquet")
        total_mbytes += df.estimated_size(unit="mb")
    return total_mbytes


def _run(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> dict:
    """Spawn a fresh interpreter to run ``run_<name>`` in ``_memory_analysis_impl.py``."""
    src = tmp_path / "src"
    src.mkdir(parents=True, exist_ok=True)
    data_size_mb = write_partitioned_measurements(src, n_partitions, rows_per_partition)

    result = subprocess.run(
        [
            sys.executable,
            str(_IMPL),
            name,
            str(tmp_path),
            str(n_partitions),
            str(rows_per_partition),
        ],
        capture_output=True,
        text=True,
        env={**os.environ},
    )
    output = (result.stdout + result.stderr).strip()
    print(f"Output from memory worker '{name}':\n{output}")

    if result.returncode != 0:
        raise RuntimeError(f"Memory worker '{name}' failed:\n{output}")
    else:
        data = json.loads(output.splitlines()[-1])
        # Some workers (e.g. by_partition_cdf) build their own data instead of
        # using src/, and report their own accurate dataset_size_mb via _check —
        # don't clobber that with the size of unrelated, unused src/ data.
        data.setdefault("dataset_size_mb", data_size_mb)
        return data


rows_per_partition = 100_000
num_partitions = 200

In [2]:
workers = ["pure_polars", "all", "by_partition", "scd2", "scd4", "by_partition_cdf"]

results = []

for i in range(1, num_partitions + 1, 20):
    for worker in workers:
        with TemporaryDirectory() as tmp_dir:
            tmp_path = Path(tmp_dir)
            result = _run(worker, tmp_path, i, rows_per_partition)
            print(
                f"[{worker}] n_partitions={i}: "
                f"Peak RSS: {result['peak_rss_mb']:.1f} MB, "
                f"Dataset size: {result['dataset_size_mb']:.1f} MB"
            )
            result.update({
                "worker": worker,
                "n_partitions": i,
                "rows_per_partition": rows_per_partition,
            })
            results.append(result)

results_df = pl.DataFrame(results)


Output from memory worker 'pure_polars':
{"peak_rss_mb": 38.734375}
[pure_polars] n_partitions=1: Peak RSS: 38.7 MB, Dataset size: 2.3 MB
Output from memory worker 'all':
{"peak_rss_mb": 76.328125}
[all] n_partitions=1: Peak RSS: 76.3 MB, Dataset size: 2.3 MB


Output from memory worker 'by_partition':
{"peak_rss_mb": 82.28125}
[by_partition] n_partitions=1: Peak RSS: 82.3 MB, Dataset size: 2.3 MB
Output from memory worker 'scd2':
{"peak_rss_mb": 90.421875}
[scd2] n_partitions=1: Peak RSS: 90.4 MB, Dataset size: 2.3 MB


Output from memory worker 'scd4':
{"peak_rss_mb": 56.40625}
[scd4] n_partitions=1: Peak RSS: 56.4 MB, Dataset size: 2.3 MB


Output from memory worker 'by_partition_cdf':
{"peak_rss_mb": 42.296875, "dataset_size_mb": 2.002716064453125}
[by_partition_cdf] n_partitions=1: Peak RSS: 42.3 MB, Dataset size: 2.0 MB


Output from memory worker 'pure_polars':
{"peak_rss_mb": 201.859375}
[pure_polars] n_partitions=21: Peak RSS: 201.9 MB, Dataset size: 49.1 MB


...

Output from memory worker 'all':
{"peak_rss_mb": 871.359375}
[all] n_partitions=181: Peak RSS: 871.4 MB, Dataset size: 438.3 MB


Output from memory worker 'by_partition':
{"peak_rss_mb": 302.890625}
[by_partition] n_partitions=181: Peak RSS: 302.9 MB, Dataset size: 438.3 MB


Output from memory worker 'scd2':
{"peak_rss_mb": 326.609375}
[scd2] n_partitions=181: Peak RSS: 326.6 MB, Dataset size: 438.3 MB


Output from memory worker 'scd4':
{"peak_rss_mb": 102.796875}
[scd4] n_partitions=181: Peak RSS: 102.8 MB, Dataset size: 438.3 MB


Output from memory worker 'by_partition_cdf':
{"peak_rss_mb": 34.265625, "dataset_size_mb": 386.5242004394531}
[by_partition_cdf] n_partitions=181: Peak RSS: 34.3 MB, Dataset size: 386.5 MB


In [3]:
import altair as alt

results_df.plot.line().encode(
    x=alt.X("n_partitions", title="n_partitions"),
    y=alt.Y("peak_rss_mb", title="peak RSS (MB, log scale)"),
    color="worker",
)


alt.Chart(...)

In [4]:
results_df.sort("worker", "n_partitions")


peak_rss_mb,dataset_size_mb,worker,n_partitions,rows_per_partition
f64,f64,str,i64,i64
76.328125,2.288818,"""all""",1,100000
261.96875,49.114227,"""all""",21,100000
370.625,96.797943,"""all""",41,100000
424.3125,144.481659,"""all""",61,100000
505.578125,192.165375,"""all""",81,100000
…,…,…,…,…
92.46875,239.944458,"""scd4""",101,100000
99.21875,289.535522,"""scd4""",121,100000
97.921875,339.126587,"""scd4""",141,100000
